In [3]:
"""
=========================================================
EXTRACCIÓN DE FEATURES — ICBHI COMPLETA (umbral 1e-5)
=========================================================
Versión idéntica al notebook original, con dos cambios:

  1. Umbral de silencio: 1e-5 en lugar de 1e-4
     (decisión basada en la auditoría de energía: el
     umbral anterior descartaba arbitrariamente grabaciones
     de tráquea/pleura con estetoscopios de menor ganancia).

  2. Mejor reporte final: distribución por clase y
     número de archivos únicos.

Parámetros del resto del pipeline (frecuencia, ventana,
features) se mantienen idénticos para asegurar
compatibilidad con los resultados previos.
=========================================================
"""

import os
import numpy as np
import librosa
import pandas as pd
from scipy.stats import skew, kurtosis

# =========================================================
# RUTAS
# =========================================================

audio_dir      = r"C:\Users\josem\Desktop\tfg\ICBHI_final_database"
diagnosis_path = r"C:\Users\josem\Desktop\tfg\dataset_con_diagnostico.csv"
output_path    = r"C:\Users\josem\Desktop\tfg\features_enriquecidas_corregido.csv"

# =========================================================
# PARÁMETROS
# =========================================================

TARGET_SR        = 4000
WINDOW_SECONDS   = 5
window_size      = TARGET_SR * WINDOW_SECONDS
N_MFCC           = 13
SILENCE_THRESHOLD = 1e-5   # <-- antes era 1e-4

# =========================================================
# CARGAR DIAGNÓSTICOS
# =========================================================

df_diag = pd.read_csv(diagnosis_path, sep=';')
diagnosis_dict = dict(zip(df_diag["patient_id"], df_diag["diagnosis"]))

print(f"Diagnósticos cargados: {len(diagnosis_dict)} pacientes")
print(f"Umbral de silencio:     {SILENCE_THRESHOLD}")
print(f"Frecuencia objetivo:    {TARGET_SR} Hz")
print(f"Ventana:                {WINDOW_SECONDS} s")
print("-" * 60)

# =========================================================
# FEATURES DE ENTROPÍA (fieles al paper)
# =========================================================

def shannon_entropy(signal):
    n      = len(signal)
    sigma  = np.std(signal)
    bw     = 3.5 * sigma / (n ** (1/3))
    bins   = max(int((np.max(signal) - np.min(signal)) / (bw + 1e-12)), 10)
    hist, _= np.histogram(signal, bins=bins, density=True)
    p      = (hist + 1e-12) / np.sum(hist + 1e-12)
    return -np.sum(p * np.log2(p))

def log_energy_entropy(signal):
    hist, _= np.histogram(signal, bins=64, density=True)
    p      = (hist + 1e-12) / np.sum(hist + 1e-12)
    return -np.sum((np.log(p)) ** 2)

def spectral_entropy(signal):
    S   = np.abs(librosa.stft(signal, n_fft=2048)) ** 2
    psd = np.sum(S, axis=1)
    psd = psd / (np.sum(psd) + 1e-12)
    return -np.sum(psd * np.log2(psd + 1e-12))

# =========================================================
# FEATURES ADICIONALES
# =========================================================
# MFCCs:           forma espectral del sonido.
# ZCR:             tasa de cruces por cero.
# RMS Energy:      energía media de la señal.
# Spectral *:      centroide, anchura y rolloff espectrales.
# Skewness/Kurt.:  forma de la distribución de amplitud.
# =========================================================

def extraer_features(window, sr):
    feats = {}

    # --- Entropías ---
    feats['shannon_entropy']    = shannon_entropy(window)
    feats['log_energy_entropy'] = log_energy_entropy(window)
    feats['spectral_entropy']   = spectral_entropy(window)

    # --- MFCCs ---
    mfccs = librosa.feature.mfcc(y=window, sr=sr, n_mfcc=N_MFCC)
    for i in range(N_MFCC):
        feats[f'mfcc_{i+1}_mean'] = np.mean(mfccs[i])
        feats[f'mfcc_{i+1}_std']  = np.std(mfccs[i])

    # --- Zero Crossing Rate ---
    zcr = librosa.feature.zero_crossing_rate(window)
    feats['zcr_mean'] = np.mean(zcr)
    feats['zcr_std']  = np.std(zcr)

    # --- RMS Energy ---
    rms = librosa.feature.rms(y=window)
    feats['rms_mean'] = np.mean(rms)
    feats['rms_std']  = np.std(rms)

    # --- Spectral Centroid ---
    sc = librosa.feature.spectral_centroid(y=window, sr=sr)
    feats['spectral_centroid_mean'] = np.mean(sc)
    feats['spectral_centroid_std']  = np.std(sc)

    # --- Spectral Bandwidth ---
    sb = librosa.feature.spectral_bandwidth(y=window, sr=sr)
    feats['spectral_bandwidth_mean'] = np.mean(sb)
    feats['spectral_bandwidth_std']  = np.std(sb)

    # --- Spectral Rolloff ---
    sr_feat = librosa.feature.spectral_rolloff(y=window, sr=sr)
    feats['spectral_rolloff_mean'] = np.mean(sr_feat)
    feats['spectral_rolloff_std']  = np.std(sr_feat)

    # --- Estadísticos temporales ---
    feats['skewness'] = skew(window)
    feats['kurtosis'] = kurtosis(window)

    return feats

# =========================================================
# PROCESAMIENTO
# =========================================================

resultados             = []
archivos_procesados    = 0
archivos_sin_diag      = []
archivos_descartados   = []   # ningún ventana válida
archivos_urti_lrti     = 0
archivos_error         = []

for file in sorted(os.listdir(audio_dir)):

    if not file.lower().endswith(".wav"):
        continue

    file_path = os.path.join(audio_dir, file)

    try:
        patient_id  = int(file.split("_")[0])
    except ValueError:
        archivos_error.append((file, "patient_id no numérico"))
        continue

    diagnostico = diagnosis_dict.get(patient_id, "Desconocido")

    if diagnostico == "Desconocido":
        archivos_sin_diag.append(file)
        continue

    if diagnostico in ["URTI", "LRTI"]:
        archivos_urti_lrti += 1
        continue

    try:
        y, sr = librosa.load(file_path, sr=TARGET_SR)

        window_count = 0

        for start in range(0, len(y), window_size):

            end    = start + window_size
            window = y[start:end]

            if len(window) < window_size:
                continue
            if np.mean(window ** 2) < SILENCE_THRESHOLD:
                continue

            feats = extraer_features(window, sr)

            feats['archivo']    = file
            feats['diagnostico']= diagnostico
            feats['ventana']    = window_count

            resultados.append(feats)
            window_count += 1

        if window_count == 0:
            archivos_descartados.append(file)
        else:
            archivos_procesados += 1

        print(f"  {file} -> {diagnostico} ({window_count} ventanas)")

    except Exception as e:
        archivos_error.append((file, str(e)))
        print(f"  Error en {file}: {e}")

# =========================================================
# RESUMEN
# =========================================================

print()
print("=" * 60)
print("RESUMEN DE EXTRACCIÓN")
print("=" * 60)
print(f"Archivos procesados con éxito:        {archivos_procesados}")
print(f"Archivos sin diagnóstico (saltados):  {len(archivos_sin_diag)}")
print(f"Archivos URTI/LRTI (excluidos):       {archivos_urti_lrti}")
print(f"Archivos sin ventanas válidas:        {len(archivos_descartados)}")
print(f"Archivos con error:                   {len(archivos_error)}")

if archivos_sin_diag:
    print(f"\nArchivos sin diagnóstico:")
    for f in archivos_sin_diag[:20]:
        print(f"   {f}")
    if len(archivos_sin_diag) > 20:
        print(f"   ... y {len(archivos_sin_diag) - 20} más")

if archivos_descartados:
    print(f"\nArchivos descartados por silencio (todas las ventanas < {SILENCE_THRESHOLD}):")
    for f in archivos_descartados:
        print(f"   {f}")

if archivos_error:
    print(f"\nArchivos con error:")
    for f, e in archivos_error:
        print(f"   {f}: {e}")

# =========================================================
# GUARDAR CSV
# =========================================================

if len(resultados) == 0:
    print("\nERROR: no se generó ninguna fila.")
else:
    df = pd.DataFrame(resultados)

    meta_cols    = ['archivo', 'diagnostico', 'ventana']
    feature_cols = [c for c in df.columns if c not in meta_cols]
    df = df[meta_cols + feature_cols]

    df.to_csv(output_path, index=False, sep=';', decimal=',')

    print(f"\n{'=' * 60}")
    print(f"CSV guardado en: {output_path}")
    print(f"Features por ventana: {len(feature_cols)}")
    print(f"Total filas (ventanas): {len(df)}")
    print(f"Archivos únicos:        {df['archivo'].nunique()}")
    print(f"\nDistribución por ventanas:\n{df['diagnostico'].value_counts()}")
    print(f"\nDistribución por archivos únicos:")
    print(df.groupby('diagnostico')['archivo'].nunique())

Diagnósticos cargados: 110 pacientes
Umbral de silencio:     1e-05
Frecuencia objetivo:    4000 Hz
Ventana:                5 s
------------------------------------------------------------
  102_1b1_Ar_sc_Meditron.wav -> Healthy (4 ventanas)
  103_2b2_Ar_mc_LittC2SE.wav -> Asthma (4 ventanas)
  104_1b1_Al_sc_Litt3200.wav -> COPD (3 ventanas)
  104_1b1_Ar_sc_Litt3200.wav -> COPD (5 ventanas)
  104_1b1_Ll_sc_Litt3200.wav -> COPD (3 ventanas)
  104_1b1_Lr_sc_Litt3200.wav -> COPD (3 ventanas)
  104_1b1_Pl_sc_Litt3200.wav -> COPD (4 ventanas)
  104_1b1_Pr_sc_Litt3200.wav -> COPD (4 ventanas)
  106_2b1_Pl_mc_LittC2SE.wav -> COPD (4 ventanas)
  106_2b1_Pr_mc_LittC2SE.wav -> COPD (4 ventanas)
  107_2b3_Al_mc_AKGC417L.wav -> COPD (4 ventanas)
  107_2b3_Ar_mc_AKGC417L.wav -> COPD (4 ventanas)
  107_2b3_Ll_mc_AKGC417L.wav -> COPD (4 ventanas)
  107_2b3_Lr_mc_AKGC417L.wav -> COPD (4 ventanas)
  107_2b3_Pl_mc_AKGC417L.wav -> COPD (4 ventanas)
  107_2b3_Pr_mc_AKGC417L.wav -> COPD (4 ventanas)
  107_2

C:\Users\josem\AppData\Local\Temp\ipykernel_28436\1934854099.py:132: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  feats['skewness'] = skew(window)
C:\Users\josem\AppData\Local\Temp\ipykernel_28436\1934854099.py:133: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  feats['kurtosis'] = kurtosis(window)


  166_1p1_Al_sc_Meditron.wav -> COPD (12 ventanas)
  166_1p1_Ar_sc_Meditron.wav -> COPD (12 ventanas)
  166_1p1_Ll_sc_Meditron.wav -> COPD (11 ventanas)
  166_1p1_Pl_sc_Meditron.wav -> COPD (12 ventanas)
  166_1p1_Pr_sc_Meditron.wav -> COPD (12 ventanas)
  167_1b1_Al_sc_Meditron.wav -> BRON (4 ventanas)
  167_1b1_Pr_sc_Meditron.wav -> BRON (4 ventanas)
  168_1b1_Al_sc_Meditron.wav -> BRON (4 ventanas)
  169_1b1_Lr_sc_Meditron.wav -> BRON (4 ventanas)
  169_1b2_Ll_sc_Meditron.wav -> BRON (4 ventanas)
  170_1b2_Al_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b2_Ar_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b2_Lr_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b2_Pl_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b2_Pr_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b2_Tc_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b3_Al_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b3_Ar_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b3_Ll_mc_AKGC417L.wav -> COPD (4 ventanas)
  170_1b3_Lr_mc_AKGC417L.wav -> COPD (4 venta